# Stresslet-Constrained RPY Brownian Dynamics — a Hands-On Tour

Simulate a suspension of **rigid spheres** with full hydrodynamic interactions
**under shear**, and watch them tumble in the flow.

This notebook is meant to be **played with**: there's a control panel near the
top, an animated movie of the suspension, a focused two-sphere demonstration of
what hydrodynamic interactions actually do, and a live slider playground at the
end. Change a number, re-run, see what happens.

---

### The physics, in three sentences

A sphere in a flowing fluid feels the flow and the flow feels the sphere — that
coupling is the **hydrodynamic interaction**. Because the spheres are *rigid*,
they cannot stretch with the local fluid strain ($E = 0$); enforcing that
constraint replaces the ordinary RPY mobility with the **stresslet-constrained**
mobility of **Fiore & Swan (2018)**,

$$R_{FU}^{-1} = M_{UF} - M_{US}\, M_{ES}^{-1}\, M_{EF}.$$

The per-particle **stresslet** $S$ (the force-dipole each rigid sphere exerts to
resist deformation) is solved for matrix-free with GMRES, and Brownian motion
consistent with the constrained fluctuation–dissipation theorem comes from the
Fiore & Swan midpoint integrator — all wrapped behind one factory,
`simulate.constrained_rpy_with_shear`.

**Along the way** we set the knobs, let the parameters auto-size, build and run,
watch the movie, see the Lees-Edwards trick, read off the stress, isolate the
hydrodynamic interaction with two spheres, and play with our own parameters.

In [ ]:
# 32-bit is plenty for a tutorial and keeps things snappy.
# (Set this to True *before* any jax_md import for production accuracy.)
from jax import config
config.update('jax_enable_x64', False)

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, colors
from matplotlib.collections import EllipseCollection, LineCollection
from matplotlib.patches import Polygon
from IPython.display import HTML

from jax_md import energy, space, simulate
from jax_md.hydro import rpy as hydro_rpy
from jax_md.hydro import stresslet_to_couplet

plt.rcParams.update({
    'figure.dpi': 130, 'savefig.dpi': 130, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.16, 'grid.linewidth': 0.8,
    'axes.facecolor': '#fbfbfd', 'figure.facecolor': 'white',
    'axes.titleweight': 'semibold', 'axes.titlesize': 12,
    'axes.edgecolor': '#888', 'xtick.color': '#444', 'ytick.color': '#444'})
plt.rcParams['animation.embed_limit'] = 80  # MB -- room for a longer movie
BLUE, MAG, TEAL, DARK, CORAL = '#2E86AB', '#A23B72', '#2A9D8F', '#264653', '#E76F51'


def add_highlight(ax, offsets, radii, scale=0.7, shift=0.26, alpha=0.35):
    """Overlay a small white off-center disk on each sphere so the flat
    coolwarm/face-colored disks read as shiny 3D balls. Returns the collection."""
    radii = np.atleast_1d(radii) * np.ones(len(offsets))
    off = np.asarray(offsets)[:, :2] + np.array([-shift, shift]) * radii[:, None]
    hl = EllipseCollection(
        widths=scale * 2 * radii, heights=scale * 2 * radii,
        angles=np.zeros(len(off)), units='xy', offsets=off,
        offset_transform=ax.transData, facecolors='white', edgecolors='none',
        alpha=alpha, zorder=6)
    ax.add_collection(hl)
    return hl


print('ready')

## 1. Control panel — tweak me!

These are the only numbers you need to touch. After changing any of them, run
the notebook top-to-bottom (Kernel -> Restart & Run All) to see the effect.

Fun things to try:
- crank **`gamma_dot`** up to shear harder (watch the box tilt faster),
- set **`kT = 0.0`** for an *athermal* run — pure shear, no Brownian jiggle,
- raise **`a`** (or **`N`**) to pack the box denser and strengthen interactions.

In [ ]:
# ============================================================
#  CONTROL PANEL
# ============================================================
N         = 64       # number of rigid spheres
L         = 12.0      # cubic box edge
a         = 1.0     # sphere (hydrodynamic) radius
eta       = 1.0      # solvent viscosity
kT        = 1.0      # temperature  (set 0.0 for athermal / no Brownian kicks)
dt        = 2e-3     # time step
gamma_dot = 10.0      # shear rate:  gamma_xy(t) = gamma_dot * t
n_steps   = 180      # number of steps to simulate
tol       = 1e-3     # target error -> AUTO Ewald-parameter selection
seed      = 0
# ============================================================

key = jax.random.PRNGKey(seed)
key, pos_key = jax.random.split(key)
base_box = jnp.eye(3) * L

# Seed on a coarse lattice + a little jitter so nothing starts overlapping.
side = int(np.ceil(N ** (1 / 3)))
grid = np.stack(np.meshgrid(*([np.arange(side)] * 3), indexing='ij'), -1)
R0 = jnp.asarray((((grid.reshape(-1, 3)[:N] + 0.5) / side)
                  + 0.01 * np.asarray(jax.random.normal(pos_key, (N, 3)))) % 1.0)

phi = N * (4 / 3) * np.pi * a ** 3 / L ** 3
print(f'N = {N}  |  box L = {L}  |  volume fraction  phi ~ {phi:.3f}')

# Lees-Edwards sheared box + a repulsive soft-sphere potential.
scalar_schedule = lambda t: gamma_dot * t
displacement, shift, box_of = space.shearing(
    base_box, shear_schedule=scalar_schedule,
    fractional_coordinates=True, remap=True)
energy_fn = energy.soft_sphere_pair(displacement, sigma=2 * a, epsilon=2.0)

## 2. Let the Ewald parameters size themselves

The split-Ewald mobility has a handful of numerical knobs: the splitting
parameter $\xi$, the real-space cutoff, the spreading support $P$, and the FFT
grid $M$. You *can* set them by hand, but you don't have to — pass a target
error `tol` and they're chosen for you by `hydro.estimate_rpy_params`.

One subtlety worth knowing: $\xi$ is **not** an accuracy knob. The exact
mobility $M = M^{(r)} + M^{(w)}$ is *independent* of $\xi$ — it only decides how
much work goes to real space vs. the FFT. So $\xi$ is picked to balance **cost**
(here, near $\xi a \approx 0.5$), while $P$, $M$, and the cutoffs are the ones
actually pinned by `tol`.

In [ ]:
est = hydro_rpy.estimate_rpy_params(
    tol=tol, A=base_box, a=a, N=N, phi=float(phi),
    shear_vector_schedule=lambda t: (gamma_dot * t, 0.0 * t, 0.0 * t),
    shear_remap=True, notes=True)

print(f'Auto-selected from tol = {tol:.0e}:')
print(f'  xi             = {est.xi:.4f}      (cost-balanced; xi*a = {est.xi * a:.3f})')
print(f'  real cutoff    = {est.rcut:.4f}')
print(f'  P  (support)   = {est.P}')
print(f'  FFT grid M     = {est.M}^3')
print(f'  k-space cutoff = {est.kcut:.4f}')
d = est.diagnostics
if d is not None:
    print(f'  error budget   : real ~{d.eps_r_est:.1e} | '
          f'wave ~{d.eps_w_est:.1e} | quad ~{d.eps_q_est:.1e}')

## 3. Build the integrator (one call) and step it

`simulate.constrained_rpy_with_shear` returns the standard JAX-MD
`(init_fn, apply_fn)` pair. We hand it the `space.shearing` 3-tuple, the energy
function, `dt`, `kT`, the physical parameters `a`/`eta`, and a *vector* shear
schedule `(gamma_xy, gamma_xz, gamma_yz)`.

Because we pass `tol` and `n_particles` (and **no** `xi`), the integrator runs
the estimator above internally and fills in every Ewald parameter for us.

In [ ]:
shear_vec = lambda t: (gamma_dot * t, 0.0 * t, 0.0 * t)

init_fn, apply_fn = simulate.constrained_rpy_with_shear(
    (displacement, shift, box_of), energy_fn, dt=dt, kT=kT,
    a=a, eta=eta, shear_vector_schedule=shear_vec,
    tol=tol, n_particles=N,                 # <-- xi, P, M, cutoffs all auto
    capacity_multiplier=2.0, extra_capacity=64)

state = init_fn(key, R0)
apply_jit = jax.jit(apply_fn)
print('initialized:', state.real_position.shape,
      '| stresslet', state.brownian_state.stresslet.shape)

## 4. Run and record the whole trajectory

We capture every frame: particle positions, each sphere's stresslet
$S_{xy}$ (for coloring the movie), the reduced Lees-Edwards strain, and the bulk
hydrodynamic stress $\sigma^H_{xy} = -\tfrac1V\sum_i S_i$.

In [ ]:
def particle_sxy(state):
    S = stresslet_to_couplet(state.brownian_state.stresslet)  # (N, 3, 3)
    return np.asarray(S[:, 0, 1])

volume = float(jnp.linalg.det(base_box))
traj, sxy_particle, times, gamma_red, sigma_xy = [], [], [], [], []

for step in range(n_steps):
    state = apply_jit(state)
    sp = particle_sxy(state)
    t = float(state.time); g_tot = gamma_dot * t
    traj.append(np.asarray(state.real_position))
    sxy_particle.append(sp)
    times.append(t)
    gamma_red.append(g_tot - np.floor(g_tot + 0.5))
    sigma_xy.append(-sp.sum() / volume)

traj = np.array(traj)                       # (T, N, 3)
sxy_particle = np.array(sxy_particle)       # (T, N)
times = np.array(times); gamma_red = np.array(gamma_red); sigma_xy = np.array(sigma_xy)

overflow = bool(state.brownian_state.rpy_state.real.neighbors.did_buffer_overflow)
assert not overflow, 'neighbor list overflowed -- raise extra_capacity'
print(f'simulated {n_steps} steps  |  total strain reached '
      f'gamma = {gamma_dot * times[-1]:.2f}  |  no overflow')

## 5. Watch it shear

Below is the suspension projected onto the **shear plane** ($x$–$y$). Each disk
is a sphere drawn to scale; its color is its instantaneous stresslet $S_{xy}$
(**red = extensional**, **blue = compressional**). The dark parallelogram is the
**Lees-Edwards box**, tilting as the strain grows; the faint grey arrows show
the imposed linear flow profile $u_x \propto y$.

Use the play / pause / scrub controls under the movie.

In [ ]:
stride = max(1, n_steps // 90)
frame_ids = list(range(0, n_steps, stride))
vmax = float(np.percentile(np.abs(sxy_particle), 98)) + 1e-9
norm = colors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

fig, ax = plt.subplots(figsize=(7.0, 6.4))
ax.set_aspect('equal'); ax.grid(False)
ax.set_xlim(-0.18 * L, 1.70 * L); ax.set_ylim(-0.18 * L, 1.18 * L)
ax.set_xlabel('x'); ax.set_ylabel('y')

# imposed shear profile u_x = gamma_dot * y: arrows that fade with height
for yv in np.linspace(0.16 * L, L, 7):
    shade = 0.86 - 0.42 * (yv / L)
    ax.annotate('', xy=(0.24 * L * (yv / L), yv), xytext=(0.0, yv),
                arrowprops=dict(arrowstyle='-|>', color=str(shade), lw=1.4))

# the tilting Lees-Edwards box: a soft fill under a crisp outline
box_fill = Polygon([[0, 0]], closed=True, facecolor=BLUE, alpha=0.05,
                   edgecolor='none', zorder=0)
box_poly = Polygon([[0, 0]], closed=True, fill=False, ec=DARK, lw=2.2,
                   joinstyle='round', zorder=4)
ax.add_patch(box_fill); ax.add_patch(box_poly)

# spheres colored by stresslet, with white highlights to read as 3D balls
coll = EllipseCollection(
    widths=2 * a * np.ones(N), heights=2 * a * np.ones(N), angles=np.zeros(N),
    units='xy', offsets=traj[0][:, :2], offset_transform=ax.transData,
    cmap='coolwarm', norm=norm, edgecolors='k', linewidths=0.5, zorder=3)
coll.set_array(sxy_particle[0])
ax.add_collection(coll)
hl = add_highlight(ax, traj[0], a)
cbar = fig.colorbar(coll, ax=ax, fraction=0.046, pad=0.03)
cbar.set_label(r'particle stresslet  $S_{xy}$   (red = extensional)')
cbar.outline.set_visible(False)
title = ax.set_title('', family='monospace', fontsize=11)

def draw(i):
    g = gamma_red[i]
    coll.set_offsets(traj[i][:, :2])
    coll.set_array(sxy_particle[i])
    hl.set_offsets(traj[i][:, :2] + np.array([-0.26 * a, 0.26 * a]))
    quad = [[0, 0], [L, 0], [L + g * L, L], [g * L, L]]
    box_poly.set_xy(quad); box_fill.set_xy(quad)
    title.set_text(f't ={times[i]:6.3f}   total g ={gamma_dot * times[i]:5.2f}'
                   f'   reduced g ={g:+.2f}')
    return coll, hl, box_poly, box_fill, title

anim = animation.FuncAnimation(fig, draw, frames=frame_ids, interval=70, blit=False)
plt.close(fig)
HTML(anim.to_jshtml(fps=14))

## 6. The Lees-Edwards trick, visualized

Notice in the movie that the box snaps back whenever it tilts too far. That's
the **strain reduction**: the *total* strain $\gamma = \dot\gamma t$ grows without
bound, but the integrator only ever feeds the mobility a **reduced** strain in
$[-0.5, 0.5)$ and remaps the particles to match. This is what keeps the
neighbor-list cell size valid over arbitrarily long runs (without it, the cells
go stale and the simulation eventually errors out).

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 3.4))
total = gamma_dot * times
ax.fill_between(total, -0.5, 0.5, color=BLUE, alpha=0.07, lw=0)
ax.axhline(0.5, ls='--', c='0.6', lw=1.0); ax.axhline(-0.5, ls='--', c='0.6', lw=1.0)
ax.plot(total, gamma_red, color=BLUE, lw=2.2, solid_capstyle='round')
ax.text(total[-1], 0.5, ' +1/2', color='0.45', va='center', fontsize=9)
ax.text(total[-1], -0.5, ' -1/2', color='0.45', va='center', fontsize=9)
ax.set_xlabel(r'total strain  $\gamma = \dot\gamma\, t$')
ax.set_ylabel(r'reduced $\gamma_{xy}$ (fed to mobility)')
ax.set_title(r'Total strain grows without bound; the reduced strain stays in $[-\frac{1}{2},\frac{1}{2})$')
fig.tight_layout(); plt.show()

## 7. Rheology: the stress the suspension carries

The hydrodynamic stress $\sigma^H_{xy}$ is the macroscopic signature of all
those microscopic stresslets. Its time-average divided by the shear rate is a
(rough, hydrodynamic) contribution to the suspension viscosity.

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 3.4))
ax.axhline(0.0, c='0.7', lw=0.8)
ax.plot(times, sigma_xy, color=MAG, lw=1.0, alpha=0.45, label=r'$\sigma^H_{xy}(t)$')
w = max(1, len(sigma_xy) // 12)
run = np.convolve(sigma_xy, np.ones(w) / w, mode='valid')
ax.plot(times[w - 1:], run, color=DARK, lw=2.4, solid_capstyle='round',
        label=f'running mean (w={w})')
half = len(sigma_xy) // 2
mean_late = float(np.mean(sigma_xy[half:]))
ax.axhline(mean_late, c=TEAL, ls='--', lw=1.4, label=f'late-time mean = {mean_late:.3f}')
ax.set_xlabel('time'); ax.set_ylabel(r'hydrodynamic shear stress $\sigma^H_{xy}$')
ax.legend(loc='best', framealpha=0.9); fig.tight_layout(); plt.show()

half = len(sigma_xy) // 2
print(f'<sigma_xy> (2nd half) / gamma_dot  ~  {np.mean(sigma_xy[half:]) / gamma_dot:.4f}')

## 8. Hydrodynamic interactions, isolated: two spheres

Here is the interaction in its purest form. We place **two** spheres in a quiet
(unsheared) box, switch off temperature, and push **only the red one** with a
constant force. The blue sphere feels **no force and no torque** — yet watch
what happens.

With hydrodynamics, the moving sphere drags fluid along, and that flow both
**translates** the blue sphere *and* **spins** it (the flow carries vorticity).
Without hydrodynamics the blue sphere would simply sit still and never turn. The
short radial line painted on each sphere tracks its rotation.

This run uses the lower-level `apply_fn.make_brownian_step(...)`, which (with
`with_torque=True`) exposes the per-step **angular velocities** in its `info`
dict — the high-level wrapper hides them.

In [ ]:
# Two spheres in a quiet (unsheared) box; deterministic (kT = 0).
L2, a2, eta2 = 10.0, 0.5, 1.0
box2 = jnp.eye(3) * L2
disp2, shift2 = space.periodic_general(box2, fractional_coordinates=True)
est2 = hydro_rpy.estimate_rpy_params(tol=1e-3, A=box2, a=a2, N=2, phi=0.0)

_, mob2 = hydro_rpy.build_rpy_mobility(
    (disp2, shift2), a=a2, xi=float(est2.xi), eta=eta2,
    rcut=float(est2.rcut), P=int(est2.P), Mgrid=int(est2.M),
    theta=float(est2.theta), lattice_extent=int(est2.lattice_extent),
    use_stresslet=True, constrained=True, with_torque=True)

# Sphere 0 (red) is pushed in +x; sphere 1 (blue) is force- and torque-free.
R_init = jnp.array([[0.30, 0.50, 0.50], [0.30, 0.62, 0.50]])
F_drive = jnp.array([[12.0, 0.0, 0.0], [0.0, 0.0, 0.0]])
force_two = lambda R, **kw: F_drive

dt2, n2 = 0.02, 80
binit2, step2 = mob2.make_brownian_step(force_two, kT=0.0, dt=dt2, mr_iters=30)
step2 = jax.jit(step2)
st2 = binit2(R_init, jax.random.PRNGKey(0))

pos_hist = [np.asarray(R_init) * L2]
theta = np.zeros(2)                 # in-plane orientation of each sphere
theta_hist = [theta.copy()]
for _ in range(n2):
    st2, info2 = step2(st2)
    omega_z = np.asarray(info2['angular_velocities'])[:, 2]
    theta = theta + omega_z * dt2
    theta_hist.append(theta.copy())
    pos_hist.append(np.asarray(st2.positions) * L2)

pos_hist = np.array(pos_hist)        # (T+1, 2, 3)
theta_hist = np.array(theta_hist)    # (T+1, 2)
d1 = pos_hist[-1, 1] - pos_hist[0, 1]
print('blue sphere (no force, no torque applied):')
print(f'  translated  dx = {d1[0]:+.3f}, dy = {d1[1]:+.3f}   (no-HI would be 0, 0)')
print(f'  rotated     theta_z = {theta_hist[-1, 1]:+.3f} rad      (no-HI would be 0)')

In [ ]:
# Static summary: trajectories (left) and the HI-induced rotation (right).
cols2 = [CORAL, BLUE]
fig, (axL, axR) = plt.subplots(1, 2, figsize=(9.6, 3.9))
axL.grid(False); axL.set_aspect('equal')
for i, lab in enumerate(['red (forced)', 'blue (force-free)']):
    axL.plot(pos_hist[:, i, 0], pos_hist[:, i, 1], color=cols2[i], lw=1.6,
             alpha=0.55, label=lab)
# draw the spheres to scale at their final pose, with spin lines + highlights
balls = EllipseCollection(
    widths=2 * a2 * np.ones(2), heights=2 * a2 * np.ones(2), angles=np.zeros(2),
    units='xy', offsets=pos_hist[-1, :, :2], offset_transform=axL.transData,
    facecolors=cols2, edgecolors='k', linewidths=0.8, zorder=3)
axL.add_collection(balls); add_highlight(axL, pos_hist[-1], a2)
for i in range(2):
    c = pos_hist[-1, i, :2]; ang = theta_hist[-1, i]
    axL.plot([c[0], c[0] + a2 * np.cos(ang)], [c[1], c[1] + a2 * np.sin(ang)],
             c='k', lw=1.8, solid_capstyle='round', zorder=5)
axL.scatter(pos_hist[0, :, 0], pos_hist[0, :, 1], facecolors='none',
            edgecolors='k', s=70, lw=1.2, zorder=6)
pad = 1.8 * a2
axL.set_xlim(pos_hist[:, :, 0].min() - pad, pos_hist[:, :, 0].max() + pad)
axL.set_ylim(pos_hist[:, :, 1].min() - pad, pos_hist[:, :, 1].max() + pad)
axL.set_xlabel('x'); axL.set_ylabel('y')
axL.set_title('trajectories (open = start, ball = end)')
axL.legend(loc='best', fontsize=9)

tt2 = np.arange(n2 + 1) * dt2
axR.plot(tt2, theta_hist[:, 0], color=CORAL, lw=2.2, label='red')
axR.plot(tt2, theta_hist[:, 1], color=BLUE, lw=2.2, label='blue (HI-induced spin)')
axR.fill_between(tt2, 0, theta_hist[:, 1], color=BLUE, alpha=0.12)
axR.axhline(0, c='0.7', lw=0.8)
axR.set_xlabel('time'); axR.set_ylabel(r'orientation $\theta_z$ (rad)')
axR.set_title('rotation'); axR.legend(loc='best', fontsize=9)
fig.tight_layout(); plt.show()

In [ ]:
# Animation: push the red sphere; the blue one is dragged and spun by the flow.
# Black radial line = orientation; fading trail = where each sphere has been.
def fading_segments(path):
    pts = np.asarray(path).reshape(-1, 1, 2)
    return np.concatenate([pts[:-1], pts[1:]], axis=1)

fids2 = list(range(0, n2 + 1, 2))
fig, ax = plt.subplots(figsize=(6.6, 5.6))
ax.set_aspect('equal'); ax.grid(False); ax.set_facecolor('#f6f9fb')
xs, ys = pos_hist[:, :, 0], pos_hist[:, :, 1]
ax.set_xlim(xs.min() - 1.8 * a2, xs.max() + 1.8 * a2)
ax.set_ylim(ys.min() - 1.8 * a2, ys.max() + 1.8 * a2)
ax.set_xlabel('x'); ax.set_ylabel('y')
cols = [CORAL, BLUE]

trail_lcs = []
for i in range(2):
    lc = LineCollection([], linewidths=2.4, zorder=2, capstyle='round')
    ax.add_collection(lc); trail_lcs.append(lc)
spheres = EllipseCollection(
    widths=2 * a2 * np.ones(2), heights=2 * a2 * np.ones(2), angles=np.zeros(2),
    units='xy', offsets=pos_hist[0, :, :2], offset_transform=ax.transData,
    facecolors=cols, edgecolors='k', linewidths=0.8, zorder=3)
ax.add_collection(spheres)
hl2 = add_highlight(ax, pos_hist[0], a2)
spin_lines = [ax.plot([], [], c='k', lw=2.0, solid_capstyle='round',
                      zorder=7)[0] for _ in range(2)]
fq = ax.quiver(pos_hist[0, 0, 0], pos_hist[0, 0, 1], 1.7 * a2, 0.0, color=CORAL,
               angles='xy', scale_units='xy', scale=1.0, width=0.015, zorder=8)
title2 = ax.set_title('', family='monospace', fontsize=11)

def draw2(k):
    p = pos_hist[k]
    spheres.set_offsets(p[:, :2])
    hl2.set_offsets(p[:, :2] + np.array([-0.26 * a2, 0.26 * a2]))
    for i in range(2):
        c = p[i, :2]; ang = theta_hist[k, i]
        spin_lines[i].set_data([c[0], c[0] + a2 * np.cos(ang)],
                               [c[1], c[1] + a2 * np.sin(ang)])
        path = pos_hist[:k + 1, i, :2]
        if len(path) > 1:
            segs = fading_segments(path)
            rgba = np.tile(np.asarray(colors.to_rgba(cols[i])), (len(segs), 1))
            rgba[:, 3] = np.linspace(0.05, 0.65, len(segs))
            trail_lcs[i].set_segments(segs); trail_lcs[i].set_color(rgba)
    fq.set_offsets(p[0, :2])
    title2.set_text(f'push red -> blue is dragged & spun   t ={k * dt2:5.2f}')
    return (spheres, hl2, fq, *spin_lines, *trail_lcs)

anim2 = animation.FuncAnimation(fig, draw2, frames=fids2, interval=90, blit=False)
plt.close(fig)
HTML(anim2.to_jshtml(fps=12))

## 9. Now you play — your own two-body experiment

The two-sphere setup is the cleanest way to *feel* a hydrodynamic interaction, so
that's what you get to drive. `two_body_demo(...)` pushes the **red** sphere with
a `force` and places the **blue** (force-free, torque-free) sphere a `gap` away —
measured in radii — at an angle `angle_deg` from the push direction. It then
reports how far the blue sphere is dragged and how much it spins. Both are
*exactly zero* without hydrodynamics, so everything you see is the interaction.
(The box auto-sizes around your choices, so the spheres never wrap into their own
periodic images.)

Below the demo, the same function is wired to **ipywidgets sliders** — drag, hit
**Run**. (Sliders need a live kernel; in a static render just call
`two_body_demo(...)` directly.)

**Physics to discover:**
- Line the blue sphere up **behind** the push (`angle_deg = 0`): the drag is
  strongest (parallel mobility exceeds perpendicular) and, by symmetry, the
  **spin is exactly zero** — a head-on wake can't twist a neighbour.
- Tilt it **off-axis** (`angle_deg = 45`): now the blue sphere clearly **spins** —
  a force only rotates a neighbour that sits off the line of the push.
- Bring it **closer** (`gap -> 2.2`, nearly touching): the coupling grows steeply.
- Push **harder** (`force` up) or **longer** (`n_steps` up) to exaggerate it.

In [ ]:
def two_body_demo(force=14.0, gap=2.6, angle_deg=45.0, a=1.0,
                  dt=0.04, n_steps=100):
    """Push the red sphere; report how the force/torque-free blue sphere is
    dragged and spun by the hydrodynamic interaction. `gap` is the centre-to-
    centre distance in radii; `angle_deg` is the separation direction relative
    to the push (+x). The box auto-sizes so the spheres never feel their own
    periodic images, and trajectories are unwrapped for a clean plot."""
    eta = 1.0
    # Size the box so both the separation and the total drift stay well inside
    # it (keeps each sphere far from its periodic images -> a clean two-body run).
    mu = 1.0 / (6.0 * np.pi * eta * a)            # isolated self-mobility
    drift = mu * force * dt * int(n_steps)         # rough red-sphere travel
    reach = max(gap * a, drift)
    L = float(max(12.0 * a, 2.5 * reach + 6.0 * a))
    box = jnp.eye(3) * L
    disp, sh = space.periodic_general(box, fractional_coordinates=True)
    est = hydro_rpy.estimate_rpy_params(tol=1e-3, A=box, a=a, N=2, phi=0.0)
    _, mob = hydro_rpy.build_rpy_mobility(
        (disp, sh), a=a, xi=float(est.xi), eta=eta, rcut=float(est.rcut),
        P=int(est.P), Mgrid=int(est.M), theta=float(est.theta),
        lattice_extent=int(est.lattice_extent),
        use_stresslet=True, constrained=True, with_torque=True)
    ang = np.deg2rad(angle_deg)
    sep_vec = gap * a * np.array([np.cos(ang), np.sin(ang), 0.0])
    c0 = np.array([0.5 * L, 0.5 * L, 0.5 * L]) - 0.5 * sep_vec
    R = jnp.asarray(np.stack([c0, c0 + sep_vec]) / L)
    Fd = jnp.array([[force, 0.0, 0.0], [0.0, 0.0, 0.0]])
    binit, step = mob.make_brownian_step(lambda r, **kw: Fd, kT=0.0, dt=dt,
                                         mr_iters=30)
    step = jax.jit(step)
    st = binit(R, jax.random.PRNGKey(0))
    pos = [np.asarray(R) * L]; th = np.zeros(2); ths = [th.copy()]
    for _ in range(int(n_steps)):
        st, info = step(st)
        th = th + np.asarray(info['angular_velocities'])[:, 2] * dt
        ths.append(th.copy()); pos.append(np.asarray(st.positions) * L)
    ths = np.array(ths)
    # Unwrap: rebuild a continuous path from minimum-image per-step displacements
    # so a sphere crossing the periodic boundary doesn't appear to teleport.
    raw = np.array(pos)                             # (T+1, 2, 3), wrapped
    steps = np.diff(raw, axis=0)
    steps -= L * np.round(steps / L)
    pos = np.concatenate([raw[:1], raw[:1] + np.cumsum(steps, axis=0)], axis=0)

    cols = [CORAL, BLUE]
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(9.6, 3.9))
    axL.grid(False); axL.set_aspect('equal')
    for i, lab in enumerate(['red (forced)', 'blue (force-free)']):
        axL.plot(pos[:, i, 0], pos[:, i, 1], color=cols[i], lw=1.6,
                 alpha=0.55, label=lab)
    balls = EllipseCollection(
        widths=2 * a * np.ones(2), heights=2 * a * np.ones(2), angles=np.zeros(2),
        units='xy', offsets=pos[-1, :, :2], offset_transform=axL.transData,
        facecolors=cols, edgecolors='k', linewidths=0.8, zorder=3)
    axL.add_collection(balls); add_highlight(axL, pos[-1], a)
    for i in range(2):
        c = pos[-1, i, :2]; aa = ths[-1, i]
        axL.plot([c[0], c[0] + a * np.cos(aa)], [c[1], c[1] + a * np.sin(aa)],
                 c='k', lw=1.8, solid_capstyle='round', zorder=5)
    axL.scatter(pos[0, :, 0], pos[0, :, 1], facecolors='none', edgecolors='k',
                s=70, lw=1.2, zorder=6)
    pad = 1.8 * a
    axL.set_xlim(pos[:, :, 0].min() - pad, pos[:, :, 0].max() + pad)
    axL.set_ylim(pos[:, :, 1].min() - pad, pos[:, :, 1].max() + pad)
    axL.set_xlabel('x'); axL.set_ylabel('y')
    axL.set_title('trajectories (open = start, ball = end)')
    axL.legend(loc='best', fontsize=9)

    tt = np.arange(int(n_steps) + 1) * dt
    axR.plot(tt, ths[:, 0], color=CORAL, lw=2.2, label='red (forced)')
    axR.plot(tt, ths[:, 1], color=BLUE, lw=2.2, label='blue (force-free)')
    axR.fill_between(tt, 0, ths[:, 1], color=BLUE, alpha=0.12)
    axR.axhline(0, c='0.7', lw=0.8)
    axR.set_xlabel('time'); axR.set_ylabel(r'orientation $\theta_z$ (rad)')
    axR.set_title('HI-induced rotation'); axR.legend(loc='best', fontsize=9)
    fig.suptitle(f'force = {force:g},  gap = {gap:g} radii,  '
                 f'angle = {angle_deg:g} deg,  a = {a:g}', fontsize=12, y=1.03)
    fig.tight_layout(); plt.show()

    dxy = pos[-1, 1, :2] - pos[0, 1, :2]
    print(f'blue sphere:  dragged dx = {dxy[0]:+.3f}, dy = {dxy[1]:+.3f}  |  '
          f'spin theta_z = {ths[-1, 1]:+.3f} rad   (both 0 without HIs)')
    return pos, ths

_ = two_body_demo()  # one quick demo run

In [ ]:
# Live playground -- drag sliders, press "Run" (needs a running kernel).
try:
    from ipywidgets import interact_manual, FloatSlider, IntSlider
    interact_manual(
        two_body_demo,
        force=FloatSlider(value=14.0, min=2.0, max=20.0, step=2.0),
        gap=FloatSlider(value=2.6, min=2.2, max=6.0, step=0.2,
                        readout_format='.1f'),
        angle_deg=FloatSlider(value=45.0, min=0.0, max=90.0, step=15.0),
        a=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.5),
        dt=FloatSlider(value=0.04, min=0.01, max=0.05, step=0.01,
                       readout_format='.2f'),
        n_steps=IntSlider(value=100, min=40, max=160, step=20))
except Exception as exc:  # pragma: no cover
    print('ipywidgets unavailable; call two_body_demo(...) directly. ', exc)

## Where to go next

- **Free (unsheared) diffusion** — use `simulate.constrained_rpy` with
  `space.periodic_general` (same arguments, no shear schedule).
- **Torques / orientations** — pass `with_torque=True` (and optionally a
  `torque_fn`) to evolve angular degrees of freedom under applied torques, as in
  the two-sphere demo above.
- **Pin parameters** — supply any of `xi`, `rcut`, `P`, `Mgrid`, `theta`,
  `lattice_extent` explicitly to override the auto-estimate; pass `xi=...` to fix
  the splitting and let only the grid be sized from `tol`.
- **Per-step diagnostics** — the high-level wrapper returns state only. For the
  solver `info` dict (Lanczos / GMRES residuals, angular velocities) drive
  `apply_fn.make_brownian_step(...)` directly, or use
  `hydro.run_brownian_chunked`, which also replays chunks automatically on
  neighbor-list overflow.
- **Inspect the chosen parameters** — call `hydro.estimate_rpy_params(..., notes=True)`
  (as in step 2) to see the full error budget before committing.

**References.** Fiore & Swan, *J. Chem. Phys.* **148**, 044114 (2018) — grand
mobility, stresslet constraint, constrained Brownian dynamics; Fiore et al.,
*J. Chem. Phys.* **146**, 124116 (2017) — the Positively Split Ewald method.